# Load Data

In [6]:
import numpy as np
import pandas as pd
import pickle
import sys
from pathlib import Path

from sklearn.linear_model import RidgeCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

from scipy.signal import find_peaks

# part_d.py — Config

In [7]:
# %%writefile part_d.py
FORMAT_VERSION = 1

# Feature engineering\n\nSingle source of truth for turning one raw `.npz` window into a feature row. Used identically by both `train` and `feature_engineering` modes.

In [8]:
# %%writefile -a part_d.py

def _stats(x, prefix, names):
    """
    Basic summary stats for a 1D array, ignoring NaNs. Returns values + appends names.
    If a window has no valid samples at all for this modality, emits np.nan rather
    than a hardcoded fallback (e.g. 0.0) — a fabricated constant would silently
    distort downstream model fitting/interpretation, whereas np.nan is handled
    explicitly and only ever imputed using train-set statistics (see cmd_train).
    """
    x = x[~np.isnan(x)]
    if x.size == 0:
        vals = [np.nan, np.nan, np.nan, np.nan, np.nan]
    else:
        vals = [
            float(np.mean(x)),
            float(np.std(x)),
            float(np.min(x)),
            float(np.max(x)),
            float(np.max(x) - np.min(x)),
        ]
    names.extend([f"{prefix}_mean", f"{prefix}_std", f"{prefix}_min", f"{prefix}_max", f"{prefix}_range"])
    return vals

def peak_intervals(x, fs, min_rate_per_min, max_rate_per_min, prominence=None):
    """
    Returns peak indices and inter-peak intervals (seconds).
    `prominence` rejects peaks that don't rise clearly above the local
    noise floor -- without it, `distance` alone lets small noise wiggles
    (e.g. T-waves, sensor noise) count as full peaks and silently double
    (or more) the detected event rate.
    """
    min_distance = int(fs * 60 / max_rate_per_min)
    peaks, _ = find_peaks(x, distance=min_distance, prominence=prominence)
    intervals = np.diff(peaks) / fs
    return peaks, intervals


def _slope(x, prefix, names):
    """Least-squares slope of a 1D array against sample index, ignoring NaNs."""
    idx = np.arange(x.shape[0])
    mask = ~np.isnan(x)
    if mask.sum() < 2:
        val = np.nan
    else:
        val = float(np.polyfit(idx[mask], x[mask], 1)[0])
    names.append(f"{prefix}_slope")
    return [val]


def _breathing_rate_features(x, fs, names, min_rate_per_min=6, max_rate_per_min=40):
    """
    Respiration-rate features from the raw breathing waveform: instantaneous
    breathing rate (from peak-to-peak timing) and breath amplitude, both
    mean/std across the window.

    NaN-safe: interpolates over isolated missing samples (the signal is
    smooth/periodic, so linear interpolation is reasonable); if too much of
    the window is missing, or too few breaths are detected to form an
    interval, emits np.nan so the downstream imputer (fit on train only)
    handles it consistently with every other feature.
    """
    prefix = "breathing"
    feature_names = [f"{prefix}_resp_rate", f"{prefix}_resp_rate_std",
                      f"{prefix}_amp_mean", f"{prefix}_amp_std"]

    x = np.asarray(x, dtype=np.float64)
    valid = ~np.isnan(x)

    if valid.sum() < 0.5 * valid.size:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    idx = np.arange(x.size)
    x_filled = np.interp(idx, idx[valid], x[valid])
    prominence = 0.5 * np.std(x_filled)

    peaks, intervals = peak_intervals(x_filled, fs=fs,
                                       min_rate_per_min=min_rate_per_min,
                                       max_rate_per_min=max_rate_per_min,
                                       prominence=prominence)

    if intervals.size < 2:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    inst_rate = 60.0 / intervals  # breaths per minute, per interval
    resp_rate = float(np.mean(inst_rate))
    resp_rate_std = float(np.std(inst_rate))

    # Amplitude: peak height above the nearest preceding trough, per breath.
    min_distance = max(1, int(fs * 60 / max_rate_per_min))
    troughs, _ = find_peaks(-x_filled, distance=min_distance, prominence=prominence)
    amps = []
    for p in peaks:
        prior_troughs = troughs[troughs < p]
        if prior_troughs.size:
            amps.append(x_filled[p] - x_filled[prior_troughs[-1]])

    amp_mean = float(np.mean(amps)) if amps else np.nan
    amp_std = float(np.std(amps)) if amps else np.nan

    names.extend(feature_names)
    return [resp_rate, resp_rate_std, amp_mean, amp_std]


def _interpolate_nans(x):
    """Linear interpolation over isolated missing samples. Returns None if
    too much of the signal is missing to interpolate meaningfully."""
    x = np.asarray(x, dtype=np.float64)
    valid = ~np.isnan(x)
    if valid.sum() < 0.5 * valid.size:
        return None
    idx = np.arange(x.size)
    return np.interp(idx, idx[valid], x[valid])


def _hrv_features(x, fs, names, min_rate_per_min=40, max_rate_per_min=200):
    """
    Heart-rate variability features from ECG: R-peak-derived heart rate,
    SDNN (overall RR-interval variability) and RMSSD (beat-to-beat
    variability). RMSSD in particular is sensitive to sympathetic/
    parasympathetic shifts -- the same autonomic response that acute
    glucose swings (especially hypoglycemia) trigger -- unlike plain
    amplitude stats on the raw ECG trace.

    NaN-safe: interpolates isolated gaps; emits np.nan if the window is
    mostly missing or too few R-peaks are detected to form >=2 RR
    intervals, so the downstream imputer (fit on train only) handles it.
    """
    prefix = "ecg"
    feature_names = [f"{prefix}_hr_mean", f"{prefix}_sdnn", f"{prefix}_rmssd"]

    x_filled = _interpolate_nans(x)
    if x_filled is None:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan]

    prominence = 0.5 * np.std(x_filled)
    peaks, rr = peak_intervals(x_filled, fs=fs,
                                min_rate_per_min=min_rate_per_min,
                                max_rate_per_min=max_rate_per_min,
                                prominence=prominence)

    if rr.size < 2:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan]

    hr_mean = float(np.mean(60.0 / rr))
    sdnn = float(np.std(rr))
    rmssd = float(np.sqrt(np.mean(np.diff(rr) ** 2)))

    names.extend(feature_names)
    return [hr_mean, sdnn, rmssd]


def _eda_scr_features(x, fs, names, tonic_window_s=20, min_amp_rel=0.5):
    """
    Skin-conductance-response (SCR) event features from EDA: how many
    sudden phasic spikes occurred in the window, and how large they were.
    Sweating (adrenergic sweat-gland activation) is a textbook acute
    hypoglycemia symptom, so SCR *events* -- not the mean EDA level -- are
    the physiologically motivated signal here.

    Tonic (slow baseline) component is estimated with a moving average and
    subtracted to isolate the phasic (fast) component, in which peaks are
    detected as SCR events. NaN-safe like the other peak-based features.
    """
    prefix = "eda"
    feature_names = [f"{prefix}_scr_count", f"{prefix}_scr_amp_mean", f"{prefix}_scr_amp_sum"]

    x_filled = _interpolate_nans(x)
    if x_filled is None:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan]

    window = max(1, int(tonic_window_s * fs))
    kernel = np.ones(window) / window
    tonic = np.convolve(x_filled, kernel, mode="same")
    phasic = x_filled - tonic

    # Relative threshold: EDA's absolute scale varies hugely by person/device
    # calibration, so a fixed absolute height cuts off very differently across
    # subjects. Scale to this window's own phasic variability instead, with a
    # tiny floor so a near-flat phasic signal doesn't trigger on pure noise.
    min_distance = max(1, int(fs * 1.0))  # SCRs don't repeat faster than ~1/s
    min_amp = max(1e-3, min_amp_rel * np.std(phasic))
    scr_peaks, _ = find_peaks(phasic, distance=min_distance, height=min_amp)

    scr_count = float(scr_peaks.size)
    if scr_peaks.size == 0:
        amp_mean, amp_sum = 0.0, 0.0
    else:
        amps = phasic[scr_peaks]
        amp_mean = float(np.mean(amps))
        amp_sum = float(np.sum(amps))

    names.extend(feature_names)
    return [scr_count, amp_mean, amp_sum]


def _bvp_pulse_features(x, fs, names, min_rate_per_min=40, max_rate_per_min=200):
    """
    Pulse-morphology features from BVP: pulse-rate variability (PRV, the
    PPG analogue of HRV), mean systolic rise time, and pulse amplitude.
    Vascular tone/blood viscosity shifts (glucose- and adrenaline-linked)
    subtly change how sharply and how strongly each pulse rises, which
    plain waveform stats don't capture.

    NaN-safe like the other peak-based features.
    """
    prefix = "bvp"
    feature_names = [f"{prefix}_prv_sdnn", f"{prefix}_prv_rmssd",
                      f"{prefix}_rise_time_mean", f"{prefix}_pulse_amp_mean"]

    x_filled = _interpolate_nans(x)
    if x_filled is None:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    min_distance = max(1, int(fs * 60 / max_rate_per_min))
    prominence = 0.5 * np.std(x_filled)
    peaks, intervals = peak_intervals(x_filled, fs=fs,
                                       min_rate_per_min=min_rate_per_min,
                                       max_rate_per_min=max_rate_per_min,
                                       prominence=prominence)

    if intervals.size < 2:
        names.extend(feature_names)
        return [np.nan, np.nan, np.nan, np.nan]

    prv_sdnn = float(np.std(intervals))
    prv_rmssd = float(np.sqrt(np.mean(np.diff(intervals) ** 2)))

    troughs, _ = find_peaks(-x_filled, distance=min_distance, prominence=prominence)
    rise_times, amps = [], []
    for p in peaks:
        prior_troughs = troughs[troughs < p]
        if prior_troughs.size:
            t = prior_troughs[-1]
            rise_times.append((p - t) / fs)
            amps.append(x_filled[p] - x_filled[t])

    rise_time_mean = float(np.mean(rise_times)) if rise_times else np.nan
    pulse_amp_mean = float(np.mean(amps)) if amps else np.nan

    names.extend(feature_names)
    return [prv_sdnn, prv_rmssd, rise_time_mean, pulse_amp_mean]


def make_features_for_window(record, feature_names_out):
    """
    record: dict-like with per-window 1D/2D arrays already sliced for ONE example,
            e.g. record["e4_bvp"] has shape (19200,), record["e4_acc"] has shape (9600, 3).
    feature_names_out: list to append feature names to (only populated on first call;
                        caller is responsible for only using this on the first row and
                        asserting consistency afterwards).
    Returns: 1D numpy array of feature values for this example.
    """
    names = []
    vals = []

    # --- BVP (E4 PPG) ---
    bvp = record["e4_bvp"]
    vals += _stats(bvp, "bvp", names)
    vals += _slope(bvp, "bvp", names)
    vals += _bvp_pulse_features(bvp, fs=64, names=names)

    # --- E4 HR (device-derived heart rate trace) ---
    hr = record["e4_hr"]
    vals += _stats(hr, "e4_hr", names)

    # --- EDA ---
    eda = record["e4_eda"]
    vals += _stats(eda, "eda", names)
    vals += _slope(eda, "eda", names)
    vals += _eda_scr_features(eda, fs=4, names=names)

    # --- Temperature ---
    temp = record["e4_temp"]
    vals += _stats(temp, "temp", names)
    vals += _slope(temp, "temp", names)

    # --- E4 accelerometer: squared magnitude a_sq(t) = ax^2+ay^2+az^2 ---
    acc = record["e4_acc"]  # (T, 3)
    acc_sq = np.nansum(acc.astype(np.float64) ** 2, axis=1)
    vals += _stats(acc_sq, "e4_acc_sq", names)

    # --- Zephyr ECG ---
    ecg = record["zephyr_ecg"]
    vals += _stats(ecg, "ecg", names)
    vals += _hrv_features(ecg, fs=250, names=names)

    # --- Zephyr accelerometer magnitude ---
    zacc = record["zephyr_acc"]
    zacc_sq = np.nansum(zacc.astype(np.float64) ** 2, axis=1)
    vals += _stats(zacc_sq, "zephyr_acc_sq", names)

    # --- Zephyr breathing ---
    breathing = record["zephyr_breathing"]
    vals += _stats(breathing, "breathing", names)
    vals += _breathing_rate_features(breathing, fs=25, names=names)

    if not feature_names_out:
        feature_names_out.extend(names)
    else:
        assert feature_names_out == names, "Feature name/order mismatch between windows"

    return np.array(vals, dtype=np.float64)

# Streaming file processing\n\nProcesses one `.npz` file at a time to keep memory bounded (indexing into an npz array still loads the full array first, so we must avoid holding many files in memory at once).

In [9]:
# %%writefile -a part_d.py

def iter_participant_files(data_dir):
    return sorted(Path(data_dir).glob("*.npz"))


def build_feature_matrix(data_dir, has_target):
    """
    Processes one .npz file at a time to keep memory bounded.
    Returns (Z, y, feature_names) where y is None if has_target is False.
    """
    feature_names = []
    feature_rows = []
    targets = [] if has_target else None

    for path in iter_participant_files(data_dir):
        with np.load(path, allow_pickle=False) as data:
            n = data["e4_bvp"].shape[0]

            fields = {
                "e4_bvp": data["e4_bvp"],
                "e4_hr": data["e4_hr"],
                "e4_eda": data["e4_eda"],
                "e4_temp": data["e4_temp"],
                "e4_acc": data["e4_acc"],
                "zephyr_ecg": data["zephyr_ecg"],
                "zephyr_acc": data["zephyr_acc"],
                "zephyr_breathing": data["zephyr_breathing"],
            }

            if has_target:
                glucose = data["glucose"]

            for i in range(n):
                record = {k: v[i] for k, v in fields.items()}
                row = make_features_for_window(record, feature_names)
                feature_rows.append(row)

            if has_target:
                targets.append(glucose)

    Z = np.vstack(feature_rows)
    y = np.concatenate(targets) if has_target else None
    return Z, y, feature_names

# Train / feature_engineering entry points

In [10]:
# %%writefile -a part_d.py

def cmd_train(protocol, train_dir, model_path):
    Z, y, feature_names = build_feature_matrix(train_dir, has_target=True)

    # Z may contain NaN columns for windows where a modality had no valid
    # samples at all (see _stats/_slope). Impute with the per-feature median,
    # fit using training data only, per the assignment's leakage rules.
    imputer = SimpleImputer(strategy="median")
    Z_imputed = imputer.fit_transform(Z)

    scaler = StandardScaler()
    Z_scaled = scaler.fit_transform(Z_imputed)

    # print corelations
    df = pd.DataFrame(Z, columns=feature_names)
    df["glucose"] = y

    corr = df.corr(numeric_only=True)["glucose"].drop("glucose")
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

    print(f"n={len(y)}  n_features={len(feature_names)}")
    print()
    print(f"{'feature':25s} {'corr':>8s}")
    for name, r in corr.head(top_n).items():
        print(f"{name:25s} {r:8.4f}")
    
    model = RidgeCV(alphas=np.logspace(-3, 3, 13))
    model.fit(Z_scaled, y)

    # Convert standardized-space coefficients back to raw feature scale so that
    # saved (intercept, coef) act directly on the *unscaled* engineered features.
    # y_hat = intercept_s + coef_s . ((z - mean) / scale)
    #       = (intercept_s - sum(coef_s * mean / scale)) + sum((coef_s/scale) * z)
    coef_scaled = model.coef_
    coef_raw = coef_scaled / scaler.scale_
    intercept_raw = model.intercept_ - np.sum(coef_scaled * scaler.mean_ / scaler.scale_)

    state = {
        "format_version": FORMAT_VERSION,
        "protocol": protocol,
        "intercept": float(intercept_raw),
        "coef": coef_raw.astype(np.float64),
        "feature_names": feature_names,
        "preprocessing_state": {
            "imputer": imputer,
        },
    }

    with open(model_path, "wb") as f:
        pickle.dump(state, f)

    print(f"[train] protocol={protocol} n={Z.shape[0]} m={Z.shape[1]} "
          f"best_alpha={model.alpha_:.4g}")


def cmd_feature_engineering(protocol, test_dir, model_path, output_path):
    with open(model_path, "rb") as f:
        state = pickle.load(f)

    assert state["protocol"] == protocol, "Model/protocol mismatch"

    Z, _, feature_names = build_feature_matrix(test_dir, has_target=False)
    assert feature_names == state["feature_names"], "Feature name/order mismatch vs. training"

    imputer = state["preprocessing_state"]["imputer"]
    Z_imputed = imputer.transform(Z)

    assert np.isfinite(Z_imputed).all(), "Non-finite values remain in final feature matrix"

    np.save(output_path, Z_imputed)
    print(f"[feature_engineering] protocol={protocol} n={Z_imputed.shape[0]} m={Z_imputed.shape[1]}")

# CLI

In [39]:
# %%writefile -a part_d.py

def main():
    if len(sys.argv) < 2:
        print("usage:\n"
              "  part_d.py train <protocol> <train_dir> <model_path>\n"
              "  part_d.py feature_engineering <protocol> <test_dir> <model_path> <output_path>")
        sys.exit(1)

    mode = sys.argv[1]

    if mode == "train":
        _, _, protocol, train_dir, model_path = sys.argv
        cmd_train(protocol, train_dir, model_path)
    elif mode == "feature_engineering":
        _, _, protocol, test_dir, model_path, output_path = sys.argv
        cmd_feature_engineering(protocol, test_dir, model_path, output_path)
    else:
        raise ValueError(f"Unknown mode: {mode}")


# if __name__ == "__main__":
#     main()

# For testing on Kaggle

In [16]:
protocol = "d1"
train_dir = "/kaggle/input/datasets/souravkumarpatel/part-d-data/Partd/random_train"
model_path = "modeld1.pkl"
test_dir = "/kaggle/input/datasets/souravkumarpatel/part-d-data/Partd/random_test"
output_path = "featuresd1"

In [19]:
protocol = "d2"
train_dir = "/kaggle/input/datasets/souravkumarpatel/part-d-data/Partd/train_set"
model_path = "modeld2.pkl"
test_dir = "/kaggle/input/datasets/souravkumarpatel/part-d-data/Partd/test_set"
output_path = "featuresd2"

Generate d2

In [32]:
import os, shutil

def safe_symlink(src, dst):
    if os.path.lexists(dst):   # lexists catches broken symlinks too, unlike exists
        os.remove(dst)
    os.symlink(src, dst)

train_d3 = "/kaggle/working/train_d3"
test_d3 = "/kaggle/working/test_d3"
os.makedirs(train_d3, exist_ok=True)
os.makedirs(test_d3, exist_ok=True)

for s in subject_train:
    safe_symlink(f"{train_set}/{s}_a.npz", f"{train_d3}/{s}_a.npz")
    safe_symlink(f"{test_set}/{s}_b.npz",  f"{train_d3}/{s}_b.npz")

for s in subject_test:
    safe_symlink(f"{train_set}/{s}_a.npz", f"{test_d3}/{s}_a.npz")
    safe_symlink(f"{test_set}/{s}_b.npz",  f"{test_d3}/{s}_b.npz")

print(len(os.listdir(train_d3)), "files in train_d3 (expect 16)")
print(len(os.listdir(test_d3)), "files in test_d3 (expect 4)")

16 files in train_d3 (expect 16)
4 files in test_d3 (expect 4)


In [41]:
# cmd_train(protocol, train_dir, model_path)

In [42]:
# cmd_feature_engineering(protocol, test_dir, model_path, output_path)

In [43]:
# !python3 part_d.py train {protocol} {train_dir} {model_path}
# !python3 part_d.py feature_engineering {protocol} {test_dir} {model_path} {output_path}

# Local evaluation (dev only)

Not part of the submission — only `part_d.py` is graded. This uses the dev test folders' `glucose` field (the real hidden test set won't have it) to compute NMAE/NMSE against your model and a median-training-target baseline, for the report's "Median baseline comparison" section.

In [17]:
def nmae(y_true, y_pred):
    y_mean = np.mean(y_true)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true - y_mean))


def nmse(y_true, y_pred):
    y_mean = np.mean(y_true)
    return np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_mean) ** 2)


def local_eval(protocol, train_dir, test_dir, model_path):
    # Train (writes model_path), exactly as the real CLI would.
    cmd_train(protocol, train_dir, model_path)

    with open(model_path, "rb") as f:
        state = pickle.load(f)

    # Median baseline needs y_train.
    _, y_train, _ = build_feature_matrix(train_dir, has_target=True)
    median_pred = np.median(y_train)

    # Dev test folders include glucose (unlike the real hidden test set) so we
    # can self-evaluate. build_feature_matrix with has_target=True reads it.
    Z_test, y_test, feature_names = build_feature_matrix(test_dir, has_target=True)
    assert feature_names == state["feature_names"], "Feature name/order mismatch vs. training"

    imputer = state["preprocessing_state"]["imputer"]
    Z_test_imputed = imputer.transform(Z_test)
    assert np.isfinite(Z_test_imputed).all(), "Non-finite values in test feature matrix"

    y_pred = state["intercept"] + Z_test_imputed @ state["coef"]
    y_pred_median = np.full_like(y_test, median_pred)

    print(f"\n=== Protocol {protocol} ===")
    print(f"n_train={len(y_train)}  n_test={len(y_test)}  n_features={len(feature_names)}")
    print(f"median(y_train) = {median_pred:.2f} mg/dL")
    print()
    print(f"{'':12s} {'NMAE':>10s} {'NMSE':>10s}")
    print(f"{'model':12s} {nmae(y_test, y_pred):10.4f} {nmse(y_test, y_pred):10.4f}")
    print(f"{'median':12s} {nmae(y_test, y_pred_median):10.4f} {nmse(y_test, y_pred_median):10.4f}")

    return state, y_test, y_pred


In [27]:
local_eval(protocol, train_dir, test_dir, model_path)
# local_eval("d2", base_dir + "train_d2", base_dir + "test_d2", "model_d2.pkl")
# local_eval("d3", base_dir + "train_d3", base_dir + "test_d3", "model_d3.pkl")

[train] protocol=d3 n=18562 m=57 best_alpha=316.2

=== Protocol d3 ===
n_train=18562  n_test=3317  n_features=57
median(y_train) = 119.00 mg/dL

                   NMAE       NMSE
model            0.9296     0.9441
median           0.9951     1.1429


({'format_version': 1,
  'protocol': 'd3',
  'intercept': 110.85080819452213,
  'coef': array([-2.18661417e+00,  6.29890988e-02, -1.12490949e-03, -3.61998113e-03,
         -7.15271167e-04,  1.19297907e+03, -1.57002964e+00, -2.76243730e-01,
          1.76397633e+01,  4.01847764e-04, -3.12224155e-03,  8.33822891e-02,
          9.18812776e-02, -5.78417649e-03, -5.59172092e-02, -1.02853476e-01,
          4.33734793e-01, -2.20766490e-03,  4.69910519e-02,  1.47411025e-01,
         -7.75456484e+01, -1.42160222e-02, -3.90891821e-01, -1.08093450e-01,
         -2.44363819e-01,  1.30061294e+00,  2.94491951e-01, -4.95422896e-02,
         -1.92440928e+00, -2.50141458e+02,  5.06596959e-03, -3.58287147e-03,
          6.42068919e-04,  1.15978669e-04,  8.96604758e-05,  2.44446899e-02,
         -1.50168010e-03,  1.50888597e-03, -2.17719527e-03, -9.64669121e-04,
         -4.22890328e-03,  4.22506896e-01, -1.92031521e-01, -1.15910129e-05,
          1.11064664e-05,  1.08630119e-06,  3.42054886e-06,  6.3000

In [31]:
with open("modeld3.pkl", "rb") as f:
    state = pickle.load(f)

imputer = state["preprocessing_state"]["imputer"]
for name, median in zip(state["feature_names"], imputer.statistics_):
    print(f"{name:25s} median={median:.4f}")


bvp_mean                  median=-0.0001
bvp_std                   median=75.8231
bvp_min                   median=-674.8200
bvp_max                   median=571.3950
bvp_range                 median=1269.8750
bvp_slope                 median=0.0000
bvp_prv_sdnn              median=0.3064
bvp_prv_rmssd             median=0.4009
bvp_rise_time_mean        median=0.3350
bvp_pulse_amp_mean        median=182.3566
e4_hr_mean                median=77.8119
e4_hr_std                 median=4.6963
e4_hr_min                 median=68.8800
e4_hr_max                 median=87.6700
e4_hr_range               median=17.4300
eda_mean                  median=0.6754
eda_std                   median=0.0537
eda_min                   median=0.4411
eda_max                   median=1.0021
eda_range                 median=0.4431
eda_slope                 median=0.0000
eda_scr_count             median=9.0000
eda_scr_amp_mean          median=0.0857
eda_scr_amp_sum           median=0.8943
temp_mean               

# Feature-target correlation diagnostic (dev only)

Not part of the submission. Quick single-feature Pearson correlation against `glucose`, computed pairwise (NaN-safe via pandas) so this can run before any imputation. Use this to see which of the 57 engineered features actually carry signal for glucose, rather than guessing which ones to refine or add next.

In [29]:
def feature_correlations(train_dir, top_n=15):
    Z, y, feature_names = build_feature_matrix(train_dir, has_target=True)

    df = pd.DataFrame(Z, columns=feature_names)
    df["glucose"] = y

    corr = df.corr(numeric_only=True)["glucose"].drop("glucose")
    corr = corr.reindex(corr.abs().sort_values(ascending=False).index)

    print(f"n={len(y)}  n_features={len(feature_names)}")
    print()
    print(f"{'feature':25s} {'corr':>8s}")
    for name, r in corr.head(top_n).items():
        print(f"{name:25s} {r:8.4f}")

    return corr


In [30]:
corr_d1 = feature_correlations(train_dir)


n=18562  n_features=57

feature                       corr
breathing_min               0.2146
breathing_mean              0.2133
breathing_max               0.2057
ecg_min                     0.1064
ecg_range                  -0.1024
ecg_max                    -0.0921
ecg_std                    -0.0756
temp_max                   -0.0655
temp_mean                  -0.0560
zephyr_acc_sq_mean         -0.0502
breathing_range            -0.0493
breathing_std              -0.0471
temp_min                   -0.0455
zephyr_acc_sq_max          -0.0397
bvp_rise_time_mean          0.0384
